# SEAL at paper scale — vector construction

Builds the SEAL reasoning-thought vectors used by the EasySteer paper's math experiment on DeepSeek-R1-Distill-Qwen-1.5B: 1000 MATH training problems (`math_train_1000.json`) are traced, each paragraph segment is classified as execution / reflection / transition by keyword, hidden states are captured only at the paragraph-break tokens, and per-category averages are exported as `execution_avg_vector.gguf` / `reflection_avg_vector.gguf` / `transition_avg_vector.gguf` for `steer.ipynb`.


In [1]:
import json
import os

from vllm import LLM, SamplingParams

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/"  # deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

# Capture needs eager execution and prefix caching off: cache-hit
# tokens are never recomputed, so their hidden states can't be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    enable_prefix_caching=False,
)

with open("math_train_1000.json", encoding="utf-8") as f:
    problems = json.load(f)

texts = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + p + "\nAssistant: <think>" for p in problems]

answers = llm.generate(
    texts,
    SamplingParams(temperature=0, max_tokens=8192, skip_special_tokens=False),
)
qa_pairs = [t + a.outputs[0].text for t, a in zip(texts, answers)]

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 04:55:50 [api_utils.py:273] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/'}


INFO 08-04 04:55:50 [model.py:623] Resolved architecture: Qwen2ForCausalLM


INFO 08-04 04:55:50 [model.py:1788] Using max model len 131072


...

Processed prompts: 100%|█████████▉| 998/1000 [06:47<00:00,  2.57it/s, est. speed input: 234.67 toks/s, output: 8694.17 toks/s]

Processed prompts: 100%|█████████▉| 999/1000 [06:48<00:00,  2.57it/s, est. speed input: 234.71 toks/s, output: 8705.91 toks/s]

Processed prompts: 100%|██████████| 1000/1000 [06:48<00:00,  2.62it/s, est. speed input: 234.72 toks/s, output: 8718.36 toks/s]

Processed prompts: 100%|██████████| 1000/1000 [06:48<00:00,  2.62it/s, est. speed input: 234.72 toks/s, output: 8718.36 toks/s]

Processed prompts: 100%|██████████| 1000/1000 [06:48<00:00,  2.45it/s, est. speed input: 234.72 toks/s, output: 8718.36 toks/s]

## Classify the paragraph breaks

Each `\n\n`-delimited reasoning segment is categorized by keyword — **transition** (switching approach), **reflection** (checking work), everything else **execution** — and attributed to the paragraph-break token that opens it.


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)

TRANSITION_KEYWORDS = [
    "alternatively", "think differently", "another way", "another approach",
    "another method", "another solution", "another strategy", "another technique",
]
REFLECTION_KEYWORDS = [
    "wait", "verify", "make sure", "hold on", "think again", "'s correct",
    "'s incorrect", "let me check", "seems right",
]


def classify(segment):
    lower = segment.lower()
    if any(k in lower for k in TRANSITION_KEYWORDS):
        return "Transition"
    if any(k in lower for k in REFLECTION_KEYWORDS):
        return "Reflection"
    return "Execution"


all_ids = []
category_by_position = []
for qa in qa_pairs:
    ids = tokenizer(qa, add_special_tokens=True).input_ids
    tokens = tokenizer.convert_ids_to_tokens(ids)
    positions = [i for i, t in enumerate(tokens) if t.endswith("ĊĊ")]
    by_pos = {}
    for j, pos in enumerate(positions):
        end = positions[j + 1] if j + 1 < len(positions) else len(ids)
        segment = tokenizer.decode(ids[pos + 1:end], skip_special_tokens=True)
        by_pos[pos] = classify(segment.strip())
    all_ids.append(ids)
    category_by_position.append(by_pos)

total = sum(len(b) for b in category_by_position)
print(f"{total} paragraph breaks across {len(qa_pairs)} traces")

92711 paragraph breaks across 1000 traces


## Capture and average

The `tokens` filter ships only the paragraph-break rows off the GPU. Prompts are passed as explicit token ids — the same ids the classifier saw — so row positions map back to categories exactly. Capture runs in chunks and accumulates per-category sums to bound client memory.


In [3]:
import numpy as np

import easysteer.hidden_states as hs
from vllm.steer_vectors.api import SelectSpec

newline_ids = sorted(
    tid for tok_str, tid in tokenizer.get_vocab().items()
    if tok_str.endswith("ĊĊ")
)

sums = {"Transition": {}, "Reflection": {}, "Execution": {}}
counts = {"Transition": 0, "Reflection": 0, "Execution": 0}

CHUNK = 200
for start in range(0, len(all_ids), CHUNK):
    chunk = all_ids[start:start + CHUNK]
    result = hs.capture(
        llm,
        [{"prompt_token_ids": ids} for ids in chunk],
        select=SelectSpec(prompt="all", tokens=newline_ids),
    )
    for i in range(len(chunk)):
        rows = result.sample(i)
        for row_idx, pos in enumerate(result.sample_positions(i)):
            # Every captured row must map to a classified break; a miss
            # means client/engine tokenization desynced.
            category = category_by_position[start + i][pos]
            counts[category] += 1
            for layer_id, tensor in rows.items():
                row = tensor[row_idx].float().numpy()
                if layer_id not in sums[category]:
                    sums[category][layer_id] = np.zeros_like(row)
                sums[category][layer_id] += row
    del result
    print(f"processed {min(start + CHUNK, len(all_ids))}/{len(all_ids)}", flush=True)

print(counts)

processed 200/1000


processed 400/1000


processed 600/1000


processed 800/1000


processed 1000/1000


{'Transition': 7501, 'Reflection': 25827, 'Execution': 59383}


In [4]:
from easysteer.steer import StatisticalControlVector

for category, per_layer in sums.items():
    control_vector = StatisticalControlVector(
        method="Average",
        directions={lid: s / counts[category] for lid, s in per_layer.items()},
        metadata={"num_vectors_averaged": counts[category]},
    )
    control_vector.export_gguf(f"{category.lower()}_avg_vector.gguf")
    print(f"{category}: averaged {counts[category]} rows")

Transition: averaged 7501 rows


Reflection: averaged 25827 rows


Execution: averaged 59383 rows
